# Set variables

In [ ]:
GROUND_TRUTH_ROOT = "vctk/"
SENSOR_ROOT = "/home/david/Documentos/gen/csv/
seed = 42

SyntaxError: EOL while scanning string literal (2767628145.py, line 2)

## Import Needed Libraries

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import *
import os
import torchinfo
import torchvision.transforms

## Import our own backend

In [ ]:
import core.nn.VCTK
from core.signal.preprocess import *
from core.nn.utils import normalize, denormalize, mapToBounds, apply_wiener

## Set seed and logging level

In [ ]:
import logging
# fix seed for reproducibility
np.random.seed(seed)
torch.manual_seed(seed)
logging.basicConfig(level=logging.INFO)

In [ ]:
Fs = 16000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
groundTruthConfig = {
    "root": GROUND_TRUTH_ROOT,
    "download": True,
}
sensorConfig = {
    "root": SENSOR_ROOT,
    "Fs": Fs,
    "device": device,
    "resampleMethod": 'sinc'
}

paired_ds = core.nn.VCTK.PairedAudioDataset(groundTruthConfig, sensorConfig)
filtered_paired_ds = core.nn.VCTK.removeOutliers(paired_ds)
train_loader, test_loader, dev_loader = core.nn.VCTK.getAudioLoaders(filtered_paired_ds, device = device, batch_size=32, ret_labels=True)

FileNotFoundError: [Errno 2] No such file or directory: '/home/david/Documentos/gen/txt'

In [ ]:
speaker_list =  paired_ds.gtDS._speaker_ids
utterance_list = list(set([e[1] for e in paired_ds.gtDS._sample_ids]))
nSpeakers = len(speaker_list)
nUtterances = len(utterance_list)
onehot_speaker = lambda x: torch.eye(nSpeakers)[speaker_list.index(x)]
onehot_utterance = lambda x: torch.eye(nUtterances)[utterance_list.index(x)]
# Is this a clean way to do this? Hell nah
# Is this efficient? Yes

In [ ]:
from core.nn.whisperPrimitives import *
device = 'cuda' if torch.cuda.is_available() else 'cpu'
flatten = nn.Flatten()
class permute (nn.Module):
    def forward(self, x):
        return x.permute(0, 2, 1)
utteranceClassifier = nn.Sequential(

).to(device)
speakerClassifier = nn.Sequential(
    nn.Conv1d(160, 128, 3),
    nn.GELU(),
    nn.Conv1d(128, 128, 3, stride = 2),
    AudioEncoder(n_mels=64, n_ctx=249, n_state=128, n_head = 8, n_layer=4),
    permute(),
    nn.GELU(),
    nn.Conv1d(128, 64, 3, stride = 2),
    nn.GELU(),
    nn.Conv1d(64, 32, 3, stride = 2),
    nn.GELU(),
    nn.Flatten(),
    nn.Linear(1952, 128),
    nn.ReLU(),
    nn.Linear(128, nSpeakers),
).to(device)

In [ ]:
import einops
tforms = core.nn.utils.buildTransforms(device = device, 
                                        Fs = Fs, 
                                        n_fft = 800, 
                                        win_length = 800,
                                        hop_length = 160,
                                        n_mels = 80,
                                        f_max_mouse = 8000,
                                        f_max_full = 8000,
                                        )
specFn, inverseSpecFn, melFilterMouse, melFilter, inverseMelFn = tforms

toMelDB = lambda batch: core.nn.utils.toMelDB(batch, specFn=specFn, mousemelfilters=melFilterMouse, fullmelfilter=melFilter)

def computeUnboundedSpec(X):
    ms = True if X.shape[1] == 2 else False
    if ms:
        X -= einops.reduce(X, "b c t -> b c ()", "mean")  # Remove mean
    else:
        X -= einops.reduce(X, "b t -> b ()", "mean")  # Remove mean
    X /= torch.std(X)                          # Normalize                                        
    X = specFn(X)                                 # Convert to mel
    X = X.abs().pow(2)                           # Power
    
    if ms:
        X = torch.einsum("bkct,fc->bkft", X, melFilterMouse)
        X = einops.rearrange(X, "b c m t -> b (m c) t", c=2)
    else:
        X = torch.einsum("bct,fc->bft", X, melFilter) # Apply mel filter
        
    X = (torch.maximum(torch.clamp(X, min=1e-10).log10(), torch.clamp(X, min=1e-10).log10().max() - 8.0) + 4.0)/ 4.0    # Convert to dB
    return X

# # The following code is used to compute the min and max values of the spectrograms

mnmn = torch.inf
mxmx = -torch.inf
for Mwav, Wwav, _, _ in dev_loader:
    Mfilt = computeUnboundedSpec(Mwav)
    M = computeUnboundedSpec(Mwav)
    W = computeUnboundedSpec(Wwav)
    mnmn = min(mnmn, M.min(), W.min(), Mfilt.min())
    mxmx = max(mxmx, M.max(), W.max(), Mfilt.max())

import einops


def computeSpec(X):
    X = computeUnboundedSpec(X)
    X = (X - mnmn) / (mxmx - mnmn)               # Map to bounds
    X = torch.clamp(X, 0, 1)                    # Clamp to [0, 1]
    X = 2 * X - 1                               # Map to [-1, 1]
    return X

def fromSpec(X):
    if X.shape[1] == 160:
        X = einops.rearrange(X, "b (m c) t -> b c m t", c=2)
        X = torch.mean(X, dim=1)
    X = (X + 1) / 2
    X = X * (mxmx - mnmn) + mnmn
    X = 10 ** (4 * X - 4)
    X = inverseMelFn(X)
    X = inverseSpecFn(X)
    return X

In [ ]:
buildClassifier = lambda n_mels, n_classes, time_len, inner_dim: nn.Sequential(
    nn.Conv1d(n_mels, 128, 3),
    nn.GELU(),
    nn.Conv1d(128, 128, 3, stride = 2),
    AudioEncoder(n_mels=64, n_ctx=(time_len // 2 - 1), n_state=128, n_head = 8, n_layer=4),
    permute(),
    nn.GELU(),
    nn.Conv1d(128, 64, 3, stride = 2),
    nn.GELU(),
    nn.Conv1d(64, 32, 3, stride = 2),
    nn.GELU(),
    nn.Flatten(),
    nn.Linear(inner_dim, 128),
    nn.ReLU(),
    nn.Linear(128, n_classes),
).to(device)

def update(model, optimizer, criterion, X, Y, class_list):
    X = X.to(device)
    X = computeSpec(X)

    Y = torch.tensor([class_list.index(i) for i in Y], device=device)
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, Y)
    loss.backward()
    optimizer.step()
    return loss.item()

N_EPOCHS = 3

In [ ]:
VCTK_GT_SPEAKER = buildClassifier(80, nSpeakers, 501, 1952)
VCTK_MS_SPEAKER = buildClassifier(160, nSpeakers, 501, 1952)

In [ ]:
def train(model, src, tgt, loader, list_of_labels, n_epochs=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    with tqdm(total = n_epochs * len(loader)) as pbar:
        for epoch in range(n_epochs):
            for (X1, X2, Y1, Y2) in train_loader:
                if src == "MS":
                    X = X1
                else:
                    X = X2
                if tgt == "SPEAKER":
                    Y = Y1
                else:
                    Y = Y2
                loss = update(model, optimizer, criterion, X, Y, list_of_labels)
                pbar.set_description(f"Loss: {loss:.4f}")
                pbar.update(1)

In [ ]:
VCTK_GT_SPEAKER.load_state_dict(torch.load("models/VCTK_GT_SPEAKER.model"))
# train(VCTK_GT_SPEAKER, "GT", "SPEAKER", train_loader, speaker_list)
torch.save(VCTK_GT_SPEAKER.state_dict(), "models/VCTK_GT_SPEAKER.model")

In [ ]:
VCTK_MS_SPEAKER.load_state_dict(torch.load("models/VCTK_MS_SPEAKER.model"))
# train(VCTK_MS_SPEAKER, "MS", "SPEAKER", train_loader, speaker_list)
torch.save(VCTK_MS_SPEAKER.state_dict(), "models/VCTK_MS_SPEAKER.model")

In [ ]:
# Get accuracy on test set
def getAccuracy(net, src, tgt, loader, class_list):
    correct = 0
    total = 0
    with torch.no_grad():
        for (X1, X2, Y1, Y2) in loader:
            if src == "MS":
                X = X1
            else:
                X = X2
            if tgt == "SPEAKER":
                Y = Y1
            else:
                Y = Y2
            X = X.to(device)
            X = computeSpec(X)
            Y = torch.tensor([class_list.index(i) for i in Y], device=device)
            outputs = net(X)
            _, predicted = torch.max(outputs.data, 1)
            total += Y.size(0)
            correct += (predicted == Y).sum().item()
    return correct / total

In [ ]:
# GT_SKR_ACC = getAccuracy(VCTK_GT_SPEAKER, "GT", "SPEAKER", test_loader, speaker_list)
# MS_SKR_ACC = getAccuracy(VCTK_MS_SPEAKER, "MS", "SPEAKER", test_loader, speaker_list)
GT_SKR_ACC = 0.7532956685499058
MS_SKR_ACC = 0.623352165725047

In [ ]:
print(f"GT Speaker Accuracy: {GT_SKR_ACC}")
print(f"MS Speaker Accuracy: {MS_SKR_ACC}")

GT Speaker Accuracy: 0.7532956685499058
MS Speaker Accuracy: 0.623352165725047


# AudioMNIST

In [ ]:
import core.nn.AudioMNIST

ds = core.nn.AudioMNIST.audioMnistDataset(device = device)
train_loader, test_loader, dev_loader = core.nn.AudioMNIST.getAudioLoaders(ds, device = device, batch_size=32, ret_labels=True)
nDigits = 10
nSpeakersMNIST = 61
digitList = list(range(10)) # Very hacky
speakerListMNIST = list(range(nSpeakersMNIST))
N_EPOCHS = 5

Train: 797 batches = 25504 samples
Test: 118 batches = 3776 samples
Dev: 24 batches = 768 samples


In [ ]:
MNIST_GT_SPEAKER = buildClassifier(80, nSpeakersMNIST, 101, 352)
MNIST_MS_SPEAKER = buildClassifier(160, nSpeakersMNIST, 101, 352)
MNIST_GT_UTTERANCE = buildClassifier(80, nDigits, 101, 352)
MNIST_MS_UTTERANCE = buildClassifier(160, nDigits, 101, 352)

In [ ]:
MNIST_GT_SPEAKER.load_state_dict(torch.load("models/MNIST_GT_SPEAKER.model"))
# train(MNIST_GT_SPEAKER, "GT", "SPEAKER", train_loader, speakerListMNIST)
torch.save(MNIST_GT_SPEAKER.state_dict(), "models/MNIST_GT_SPEAKER.model")

In [ ]:
MNIST_MS_SPEAKER.load_state_dict(torch.load("models/MNIST_MS_SPEAKER.model"))
# train(MNIST_MS_SPEAKER, "MS", "SPEAKER", train_loader, speakerListMNIST)
torch.save(MNIST_MS_SPEAKER.state_dict(), "models/MNIST_MS_SPEAKER.model")

In [ ]:
MNIST_GT_UTTERANCE.load_state_dict(torch.load("models/MNIST_GT_UTTERANCE.model"))
# train(MNIST_GT_UTTERANCE, "GT", "UTTERANCE", train_loader, digitList)
torch.save(MNIST_GT_UTTERANCE.state_dict(), "models/MNIST_GT_UTTERANCE.model")

In [ ]:
MNIST_MS_UTTERANCE.load_state_dict(torch.load("models/MNIST_MS_UTTERANCE.model"))
# train(MNIST_MS_UTTERANCE, "MS", "UTTERANCE", train_loader, digitList)
torch.save(MNIST_MS_UTTERANCE.state_dict(), "models/MNIST_MS_UTTERANCE.model")

In [ ]:
MNIST_GT_SPEAKER_ACC = getAccuracy(MNIST_GT_SPEAKER, "GT", "SPEAKER", test_loader, speakerListMNIST)
MNIST_MS_SPEAKER_ACC = getAccuracy(MNIST_MS_SPEAKER, "MS", "SPEAKER", test_loader, speakerListMNIST)
MNIST_GT_UTTERANCE_ACC = getAccuracy(MNIST_GT_UTTERANCE, "GT", "UTTERANCE", test_loader, digitList)
MNIST_MS_UTTERANCE_ACC = getAccuracy(MNIST_MS_UTTERANCE, "MS", "UTTERANCE", test_loader, digitList)

In [ ]:
print(f"GT Speaker Accuracy: {MNIST_GT_SPEAKER_ACC}")
print(f"MS Speaker Accuracy: {MNIST_MS_SPEAKER_ACC}")
print(f"GT Utterance Accuracy: {MNIST_GT_UTTERANCE_ACC}")
print(f"MS Utterance Accuracy: {MNIST_MS_UTTERANCE_ACC}")

GT Speaker Accuracy: 0.9288
MS Speaker Accuracy: 0.8005333333333333
GT Utterance Accuracy: 0.9912
MS Utterance Accuracy: 0.7573333333333333
